In [1]:
%pip install azure-eventhub

StatementMeta(, 4d384c57-e942-4489-8ac3-d701aa10b4a1, 7, Finished, Available, Finished, True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 5.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 18.1 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Not uninstalling typing-extensions at /home/trusted-service-user/cluster-env/trident_env/lib/python3.10/site-packages, outside environment /nfs4/pyenv-61e51b89-9fc8-4d7e-9775-6e912ab65f1d
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 2.0.0 requires sentencepiece, which is not installed.
sentence-transformers 2.0.0 requires torchvision, which is not installed.
dash 2.14.0 requires Flask<2.3.0,>=1.0.4, but you have flask 3.0.0 which is incompatible.
dash 2.14.0 requires Werkzeug<2.3.0, but you hav

In [ ]:
import time
import uuid
import json
import random
from datetime import datetime
from azure.eventhub import EventHubProducerClient, EventData


# -------------------------------------
# FABRIC EVENTSTREAM CONNECTION
# -------------------------------------
EVENT_HUB_CONNECTION_STRING = "xxx"

# -------------------------------------
# SPAIN SUPERMARKET LOCATIONS
# -------------------------------------
# Major warehouse and supermarket locations across Spain
WAREHOUSES = [
    {"id": "WH_MAD", "name": "Madrid Central Warehouse", "coords": [-3.7038, 40.4168]},
    {"id": "WH_BCN", "name": "Barcelona Distribution Center", "coords": [2.1734, 41.3851]},
    {"id": "WH_VAL", "name": "Valencia Logistics Hub", "coords": [-0.3763, 39.4699]},
    {"id": "WH_SEV", "name": "Seville Warehouse", "coords": [-5.9845, 37.3891]},
]

REPAIRING_OFFICES = [
    {"id": "RO_MAD", "name": "Madrid Central Repair Office",    "coords": [-3.7038, 40.4168]},
    {"id": "RO_BCN", "name": "Barcelona Maintenance Hub",       "coords": [2.1734,  41.3851]},
    {"id": "RO_SEV", "name": "Seville Field Operations Base",   "coords": [-5.9845, 37.3891]},
    {"id": "RO_BIL", "name": "Bilbao Technical Service Center", "coords": [-2.9253, 43.2627]},
]

SUPERMARKETS = [
    {"id": "SM_001", "name": "Mercadona Madrid Norte", "coords": [-3.6908, 40.4637]},
    {"id": "SM_002", "name": "Carrefour Barcelona Centre", "coords": [2.1686, 41.3874]},
    {"id": "SM_003", "name": "Dia Valencia Centro", "coords": [-0.3691, 39.4765]},
    {"id": "SM_004", "name": "Lidl Seville Este", "coords": [-5.9515, 37.3778]},
    {"id": "SM_005", "name": "Alcampo Zaragoza", "coords": [-0.8773, 41.6488]},
    {"id": "SM_006", "name": "Mercadona Malaga", "coords": [-4.4214, 36.7213]},
    {"id": "SM_007", "name": "Carrefour Bilbao", "coords": [-2.9253, 43.2627]},
    {"id": "SM_008", "name": "Dia Alicante", "coords": [-0.4907, 38.3452]},
    {"id": "SM_009", "name": "Lidl Granada", "coords": [-3.5986, 37.1773]},
    {"id": "SM_010", "name": "Mercadona Murcia", "coords": [-1.1307, 37.9922]},
    {"id": "SM_011", "name": "Carrefour Valladolid", "coords": [-4.7245, 41.6523]},
    {"id": "SM_012", "name": "Dia Cordoba", "coords": [-4.7792, 37.8882]},
    {"id": "SM_013", "name": "Lidl Salamanca", "coords": [-5.6640, 40.9701]},
    {"id": "SM_014", "name": "Alcampo Pamplona", "coords": [-1.6440, 42.8125]},
    {"id": "SM_015", "name": "Mercadona Toledo", "coords": [-4.0273, 39.8628]},
]

SUBSTATIONS = [
    {"id": "SUB_001", "name": "Madrid Solar Park",        "coords": [-3.6345, 40.5423]},
    {"id": "SUB_002", "name": "Barcelona Wind Farm",      "coords": [2.1123,  41.4231]},
    {"id": "SUB_003", "name": "Valencia Marine Station",  "coords": [-0.3318, 39.4561]},
    {"id": "SUB_004", "name": "Zaragoza Solar Array",     "coords": [-0.9534, 41.7342]},
    {"id": "SUB_005", "name": "Seville Hydro Plant",      "coords": [-5.7234, 37.5443]},
    {"id": "SUB_006", "name": "Bilbao Wind Turbines",     "coords": [-2.6534, 43.3213]},
    {"id": "SUB_007", "name": "Almeria Solar Complex",    "coords": [-2.3521, 37.0567]},
    {"id": "SUB_008", "name": "Galicia Marine Array",     "coords": [-8.4115, 43.3623]},
    {"id": "SUB_009", "name": "Malaga Solar Farm",        "coords": [-4.4936, 36.6885]},
    {"id": "SUB_010", "name": "Murcia Wind Park",         "coords": [-1.2345, 38.1623]},
    {"id": "SUB_011", "name": "Toledo Hydro Station",     "coords": [-4.3521, 39.7834]},
    {"id": "SUB_012", "name": "Granada Solar Plant",      "coords": [-3.5678, 36.9234]},
    {"id": "SUB_013", "name": "Asturias Wind Complex",    "coords": [-5.9123, 43.2456]},
    {"id": "SUB_014", "name": "Tarragona Marine Station", "coords": [1.2444,  41.1167]},
    {"id": "SUB_015", "name": "Salamanca Solar Array",    "coords": [-5.6789, 40.9943]},
    {"id": "SUB_016", "name": "Navarra Wind Farm",        "coords": [-1.5234, 42.3667]},
    {"id": "SUB_017", "name": "Extremadura Hydro Plant",  "coords": [-6.5234, 39.7223]},
    {"id": "SUB_018", "name": "Cantabria Marine Array",   "coords": [-3.8099, 43.4623]},
    {"id": "SUB_019", "name": "Cordoba Solar Complex",    "coords": [-4.7796, 37.8845]},
    {"id": "SUB_020", "name": "Leon Wind Turbines",       "coords": [-5.5678, 42.8234]},
]

# Spanish driver names
DRIVER_NAMES = [
    "Carlos García", "María López", "José Martínez", "Ana Rodríguez", "Miguel Fernández",
    "Laura González", "Antonio Sánchez", "Carmen Pérez", "Francisco Romero", "Isabel Ruiz",
    "Manuel Torres", "Elena Navarro", "Javier Jiménez", "Sofía Moreno", "David Muñoz",
    "Patricia Álvarez", "Pedro Castillo", "Lucía Ortega", "Alberto Ramos", "Cristina Serrano"
]

TECHNICIANS = [
    "Carlos García Romero", "Ainhoa Etxeberria Zubiaurre", "María Fernández López", "Jorge Navarro Blanco", "Laura Castillo Vega",  
    "Rubén Morales Sanz", "Iñigo Urresti Arizmendi", "Carmen Delgado Ruiz", "Alejandro Torres Prado", "Pilar Santos Aguilar", 
    "David Herrera Montes", "Eva Serrano Costa"
]

# -------------------------------------
# TRUCK SIMULATOR CLASS
# -------------------------------------
class TruckSimulator:
    
    def __init__(self, truck_id: str, warehouse: dict, destination: dict, driver: str, producer_client):
        self.truck_id = truck_id
        self.warehouse = warehouse
        self.destination = destination
        self.driver = driver
        self.producer_client = producer_client
        
        # Initialize route
        self.current_coords = warehouse["coords"].copy()
        self.target_coords = destination["coords"]
        
        # Calculate total distance and steps
        self.progress = 0.0  # 0.0 to 1.0
        self.speed_kmh = random.uniform(60, 90)  # Speed in km/h
        
        # Truck characteristics
        self.max_load_kg = random.choice([12000, 15000, 18000, 20000])
        self.current_load_kg = random.randint(int(self.max_load_kg * 0.6), self.max_load_kg)
        
        # Status
        self.status = "in_transit"
        self.delivery_started = datetime.now().isoformat()
        
    def calculate_next_position(self):
        """Calculate next GPS position along the route with realistic road movement."""
        if self.progress >= 1.0:
            self.status = "delivered"
            self.current_coords = self.target_coords
            return
        
        # Increment progress (simulate time-based movement)
        # Add some randomness to simulate traffic, road conditions
        progress_increment = random.uniform(0.02, 0.05)
        self.progress = min(1.0, self.progress + progress_increment)
        
        # Linear interpolation between origin and destination
        # In production, you'd use actual road routing APIs
        lat_start, lon_start = self.warehouse["coords"][1], self.warehouse["coords"][0]
        lat_end, lon_end = self.target_coords[1], self.target_coords[0]
        
        # Calculate current position with some realistic "road wiggle"
        base_lon = lon_start + (lon_end - lon_start) * self.progress
        base_lat = lat_start + (lat_end - lat_start) * self.progress
        
        # Add small random deviations to simulate following roads (not straight line)
        wiggle = 0.005 * random.uniform(-1, 1)
        self.current_coords = [
            round(base_lon + wiggle, 6),
            round(base_lat + wiggle, 6)
        ]
    
    def generate_truck_reading(self):
        """Generate GeoJSON Feature for truck's current position."""
        self.calculate_next_position()
        
        # Calculate estimated arrival time
        distance_remaining = (1 - self.progress) * 100  # Simplified distance in km
        eta_minutes = int((distance_remaining / self.speed_kmh) * 60)
        
        # Create GeoJSON Feature
        geojson_feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": self.current_coords  # [longitude, latitude]
            },
            "properties": {
                "reading_id": str(uuid.uuid4()),
                "timestamp": datetime.now().isoformat(),
                "truck_id": self.truck_id,
                "driver_name": self.driver,
                "origin_warehouse_id": self.warehouse["id"],
                "origin_warehouse_name": self.warehouse["name"],
                "destination_supermarket_id": self.destination["id"],
                "destination_supermarket_name": self.destination["name"],
                "current_load_kg": self.current_load_kg,
                "max_load_kg": self.max_load_kg,
                "load_percentage": round((self.current_load_kg / self.max_load_kg) * 100, 1),
                "speed_kmh": round(self.speed_kmh, 1),
                "status": self.status,
                "progress_percentage": round(self.progress * 100, 1),
                "eta_minutes": eta_minutes,
                "delivery_started": self.delivery_started,
                "fuel_level_percent": round(random.uniform(40, 95), 1),
                "temperature_cargo_c": round(random.uniform(2, 6), 1)  # Refrigerated cargo
            }
        }
        
        return geojson_feature
    
    def send_to_fabric(self, geojson_data):
        """Send GeoJSON event to Microsoft Fabric Eventstream."""
        try:
            event_data_batch = self.producer_client.create_batch()
            event_data_batch.add(EventData(json.dumps(geojson_data)))
            self.producer_client.send_batch(event_data_batch)
            
            status_icon = "🚚" if self.status == "in_transit" else "✅"
            print(f"{status_icon} {self.truck_id} → {self.progress*100:.1f}% to {self.destination['name']}")
            
        except Exception as e:
            print(f"❌ Failed to send event: {e}")
    
    def process_reading(self):
        """Generate and send truck reading."""
        reading = self.generate_truck_reading()
        self.send_to_fabric(reading)
        return self.status == "delivered"


# -------------------------------------
# TRUCK FLEET MANAGER
# -------------------------------------
class TruckFleetManager:
    
    def __init__(self, producer_client, num_trucks=15):
        self.producer_client = producer_client
        self.active_trucks = []
        self.num_trucks = num_trucks
        
    def create_new_delivery(self):
        """Create a new truck delivery route."""
        truck_id = f"TRK_{random.randint(100, 999)}"
        warehouse = random.choice(REPAIRING_OFFICES)
        destination = random.choice(SUBSTATIONS)
        driver = random.choice(TECHNICIANS)
        
        truck = TruckSimulator(truck_id, warehouse, destination, driver, self.producer_client)
        return truck
    
    def manage_fleet(self):
        """Maintain active fleet of trucks."""
        # Initialize fleet
        for _ in range(self.num_trucks):
            self.active_trucks.append(self.create_new_delivery())
        
        print(f"🚚 Fleet initialized with {self.num_trucks} active trucks")
        print("=" * 70)
        
        while True:
            # Process each truck
            completed_trucks = []
            
            for truck in self.active_trucks:
                is_delivered = truck.process_reading()
                if is_delivered:
                    completed_trucks.append(truck)
                    print(f"📦 {truck.truck_id} completed delivery to {truck.destination['name']}")
            
            # Replace completed trucks with new deliveries
            for truck in completed_trucks:
                self.active_trucks.remove(truck)
                self.active_trucks.append(self.create_new_delivery())
                print(f"🆕 New delivery route assigned: {self.active_trucks[-1].truck_id}")
            
            # Wait before next update cycle
            time.sleep(random.uniform(2, 5))


# -------------------------------------
# MAIN FUNCTION
# -------------------------------------
def main():
    
    # Connect to Event Hub
    try:
        producer = EventHubProducerClient.from_connection_string(
            conn_str=EVENT_HUB_CONNECTION_STRING
        )
        print("✅ Connected to Fabric Event Stream")
    except Exception as e:
        print(f"❌ Failed to connect to Event Stream: {e}")
        print("Please update EVENT_HUB_CONNECTION_STRING with your credentials!")
        return
    
    print("🚀 Spain Supermarket Truck Tracker - Real-time GeoJSON Simulator")
    print("📍 Tracking deliveries across Spain")
    print("=" * 70)
    
    # Start fleet management
    try:
        fleet_manager = TruckFleetManager(producer, num_trucks=15)
        fleet_manager.manage_fleet()
    
    except KeyboardInterrupt:
        print("\n⏹ Stopping truck tracker...")
    
    finally:
        producer.close()
        print("✅ Connection closed")


if __name__ == "__main__":
    main()

StatementMeta(, 4d384c57-e942-4489-8ac3-d701aa10b4a1, 9, Submitted, Running, Running, True)

✅ Connected to Fabric Event Stream
🚀 Spain Supermarket Truck Tracker - Real-time GeoJSON Simulator
📍 Tracking deliveries across Spain
🚚 Fleet initialized with 15 active trucks
🚚 TRK_846 → 3.3% to Toledo Hydro Station
🚚 TRK_992 → 4.1% to Bilbao Wind Turbines
🚚 TRK_216 → 2.1% to Granada Solar Plant
🚚 TRK_943 → 2.4% to Salamanca Solar Array
🚚 TRK_765 → 3.5% to Malaga Solar Farm
🚚 TRK_765 → 2.7% to Barcelona Wind Farm
🚚 TRK_876 → 2.5% to Seville Hydro Plant
🚚 TRK_779 → 4.5% to Granada Solar Plant
🚚 TRK_762 → 5.0% to Granada Solar Plant
🚚 TRK_414 → 2.3% to Cordoba Solar Complex
🚚 TRK_104 → 2.5% to Valencia Marine Station
🚚 TRK_451 → 2.3% to Leon Wind Turbines
🚚 TRK_818 → 2.1% to Cordoba Solar Complex
🚚 TRK_497 → 3.0% to Tarragona Marine Station
🚚 TRK_436 → 3.4% to Leon Wind Turbines
🚚 TRK_846 → 7.1% to Toledo Hydro Station
🚚 TRK_992 → 7.7% to Bilbao Wind Turbines
🚚 TRK_216 → 5.6% to Granada Solar Plant
🚚 TRK_943 → 6.4% to Salamanca Solar Array
🚚 TRK_765 → 6.4% to Malaga Solar Farm
🚚 TRK_765